In [2]:
import os
import sys
import json
import warnings
import numpy as np
import sympy as sp
import pandas as pd
import proplot as pplt
from IPython.display import display,HTML
sys.path.insert(0,'..')
warnings.filterwarnings('ignore')
pplt.rc.update({
    'tick.minor':False,
    'savefig.dpi':300,
    'font.size':9,
    'label.size':9,
    'tick.labelsize':9,
    'legend.fontsize':9})

In [3]:
with open('../scripts/configs.json','r',encoding='utf-8') as f:
    CONFIGS = json.load(f)
MODELSDIR = CONFIGS['filepaths']['models']
SRCONFIG  = CONFIGS['experiments']['sr']
SEEDS     = SRCONFIG['seeds']

In [4]:
VARDICT = {
    'rh':r'\widehat{\mathrm{RH}}',
    'thetae':r'\widehat{\theta}_{e}',
    'thetaestar':r'{\widehat{\theta}_{e}^{*}}',
    'lf':r'\mathrm{LF}',
    'shf':r'\mathrm{SHF}',
    'lhf':r'\mathrm{LHF}',}

SYMBOLS  = {k:sp.Symbol(k) for k in VARDICT}

FUNCDICT = {
    'cube':lambda x:x**3,
    'square':lambda x:x**2,
    'sqrt':sp.sqrt,
    'abs':sp.Abs,
    'neg':lambda x:-x,
    'exp':sp.exp,
    'log':sp.log,
    'sin':sp.sin,
    'cos':sp.cos}

LATEXREPLACE = {
    SYMBOLS['rh']:sp.Symbol(r'\widehat{\mathrm{RH}}'),
    SYMBOLS['thetae']:sp.Symbol(r'\widehat{\theta}_{e}'),
    SYMBOLS['thetaestar']:sp.Symbol(r'\widehat{\theta}_{e}^{*}'),
    SYMBOLS['lf']:sp.Symbol(r'\mathrm{LF}'),
    SYMBOLS['shf']:sp.Symbol(r'\mathrm{SHF}'),
    SYMBOLS['lhf']:sp.Symbol(r'\mathrm{LHF}')}

TERMORDER = {
    'rh':0,
    'thetae':1,
    'thetaestar':2,
    'lf':3,
    'shf':4,
    'lhf':5}

def _to_sympy_expr(eq):
    return sp.sympify(eq,locals={**SYMBOLS,**FUNCDICT})

def _round_numbers(expr,ndigits=4):
    return expr.xreplace({n: sp.Float(round(float(n),ndigits),ndigits) for n in expr.atoms(sp.Float)})

def _term_key(term):
    symbols = term.free_symbols
    if not symbols:
        return (99, str(term))
    names = sorted(s.name for s in symbols)
    return (min(TERMORDER.get(n,50) for n in names),str(term))

def _ordered_add_terms(expr):
    if isinstance(expr,sp.Add):
        terms = sp.Add.make_args(expr)
        return sp.Add(*sorted(terms,key=_term_key),evaluate=False)
    return expr

def _order_expr(expr):
    if expr.args:
        expr = expr.func(*[_order_expr(arg) for arg in expr.args],evaluate=False)
    if isinstance(expr,sp.Add):
        expr = _ordered_add_terms(expr)
    return expr

def _latex_expr(expr):
    SYMBOLNAMES = {SYMBOLS[k]: v for k, v in VARDICT.items()}
    latex = sp.latex(expr,symbol_names=SYMBOLNAMES,mul_symbol='dot')
    latex = latex.replace(r'\left', '').replace(r'\right', '')
    latex = ' '.join(latex.split())
    varslatex = list(VARDICT.values())
    for a in varslatex:
        for b in varslatex:
            latex = latex.replace(f'{a} \\cdot {b}',f'{a}({b})')
    return latex

def prettify(eq):
    try:
        expr = _to_sympy_expr(str(eq).strip())
        expr = sp.expand(expr)
        expr = sp.simplify(expr)
        expr = _round_numbers(expr, ndigits=4)
        expr = _order_expr(expr)
        return '$'+_latex_expr(expr)+'$'
    except Exception:
        return str(eq).strip()
        
def load_equations(runname):
    seedframes = {}
    for seed in SEEDS:
        filepath = os.path.join(MODELSDIR, 'sr', f'{runname}_{seed}_equations.csv')
        df = pd.read_csv(filepath)
        df['seed'] = seed
        seedframes[seed] = df
    return seedframes
    
def equation_table(runname):
    seedframes = load_equations(runname)
    if not seedframes:
        return pd.DataFrame()
    rows = []
    for seed,df in seedframes.items():
        for _,row in df.iterrows():
            rows.append({'Seed':seed,'Complexity':int(row['complexity']),
                         'Loss':float(row['loss']),'Equation':prettify(str(row['equation']))})
    return pd.DataFrame(rows).sort_values(['Seed','Complexity']).reset_index(drop=True)

In [5]:
for runname in SRCONFIG['runs']:
    tbl = equation_table(runname)
    display(HTML(tbl.to_html(escape=False,index=False)))

Seed,Complexity,Loss,Equation
42,1,0.998213,$0.0017$
42,2,0.679273,$\widehat{\mathrm{RH}}$
42,4,0.650225,$\widehat{\mathrm{RH}}^{3}$
42,6,0.631149,$1.339 \cdot \widehat{\mathrm{RH}}^{3}$
42,8,0.622777,$\widehat{\mathrm{RH}}^{3} + 0.4826 \cdot \widehat{\mathrm{RH}}^{2} + 0.0776 \cdot \widehat{\mathrm{RH}} - 0.183$
42,9,0.603885,$\widehat{\mathrm{RH}} - |{\mathrm{SHF} + 0.4472}| + 0.6026$
42,10,0.579758,$\widehat{\mathrm{RH}} + \widehat{\theta}_{e} - {\widehat{\theta}_{e}^{*}} - 1.086$
42,11,0.500468,$4.3 \cdot \widehat{\theta}_{e} - 6.177 \cdot {\widehat{\theta}_{e}^{*}} - 4.714$
42,13,0.493033,$- \mathrm{LF}(\widehat{\theta}_{e}) + 1.415 \cdot \mathrm{LF}({\widehat{\theta}_{e}^{*}}) + 1.091 \cdot \mathrm{LF} + 4.761 \cdot \widehat{\theta}_{e} - 6.734 \cdot {\widehat{\theta}_{e}^{*}} - 5.195$
42,14,0.484834,$\widehat{\mathrm{RH}}(\widehat{\theta}_{e}) - 1.436 \cdot \widehat{\mathrm{RH}}({\widehat{\theta}_{e}^{*}}) - 1.081 \cdot \widehat{\mathrm{RH}} + 3.383 \cdot \widehat{\theta}_{e} - 4.857 \cdot {\widehat{\theta}_{e}^{*}} - 3.657$


In [ ]:
COLORS = ['#5BA7DA','#F2C85E','#D42028']
BEST   = {
    42:[11,16,25],
    72:[11,16,25],
    102:[11,16]}

for runname in SRCONFIG['runs']:
    seedframes = load_equations(runname)
    fig,ax = pplt.subplots(refwidth=5,refheight=2)
    ax.format(xlabel='Equation Complexity',xlim=(0,26),ylabel='Training MSE')
    for i,(seed,df) in enumerate(sorted(seedframes.items())):
        df      = df.sort_values('complexity')
        color   = COLORS[i%len(COLORS)]
        ax.plot(df['complexity'],df['loss'],color=color,alpha=0.5,linewidth=1,linestyle='--',zorder=1,label='')
        ax.scatter(df['complexity'],df['loss'],color=color,marker='.',zorder=3,label=f'Seed {seed}')
        bestcs = BEST.get(seed)
        if bestcs is not None:
            if not isinstance(bestcs,(list,tuple,set)):
                bestcs = [bestcs]
            for bestc in bestcs:
                row = df[df['complexity']==bestc]
                if not row.empty:
                    ax.scatter([row['complexity'].values[0]],[row['loss'].values[0]],
                        color=color,edgecolors='k',marker='*',markersize=100,linewidths=0.5,zorder=5)  
    ax.scatter([],[],color='k',marker='*',markersize=100,label='Selected Best')
    ax.legend(loc='ur',ncols=1)
    pplt.show()
    fig.save('../figs/fig_S2.jpg')